In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
catalog = dbutils.widgets.get("catalog")
checkpoints_dir = dbutils.widgets.get("checkpoints_dir")

In [ ]:
import os
import sys

home = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

sys.path.append(home)

from src.utils.json_parser import parse_json
from src.utils.cleaner import good_sales_records, bad_sales_records
from src.utils.time_utils import date_day_weekOfMonth

In [ ]:
from pyspark.sql.types import StructField, IntegerType, StringType, StructType, TimestampType, LongType
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# define schema for sales
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("sales_id", LongType(), True),
    StructField("employee_id", LongType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", LongType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

df = spark.readStream.table(f"{catalog}.brz.sales_raw")

parsed_df = parse_json(spark, df, "value", schema)

parsed_df = (parsed_df.select("parsed.sales_id", "parsed.employee_id", "parsed.region_id",
                              "parsed.product_id", "parsed.quantity", "parsed.sales_amount",
                              "parsed.event_time"))

def process_sales(batch_df, batch_id):
    
    w = Window.partitionBy(F.col("sales_id")).orderBy(F.col("event_time").desc())

    # deduplicate sales events

    batch_df = (batch_df.withColumn("rn", F.row_number().over(w))
                        .filter(F.col("rn")==1)
                        .drop("rn"))

    # process good sales using utils

    good_sales_df = good_sales_records(spark, batch_df)

    # process bad sales using utils

    bad_sales_df = bad_sales_records(spark, batch_df)

    # add date utils to good sales

    good_sales_df = date_day_weekOfMonth(spark, good_sales_df, "event_time")

    good_sales_df = (good_sales_df.select("sales_id", "employee_id", "region_id", "product_id",
                                        "quantity", "sales_amount", "event_time", "event_date",
                                        "day", "week_of_month", "processed_time"))
    
    good_sales_df.write.format("delta").mode("append").partitionBy("event_date").saveAsTable(f"{catalog}.slv.sales")

    bad_sales_df.write.format("delta").mode("append").saveAsTable(f"{catalog}.brz.bad_sales_records")

query = (parsed_df.writeStream
                    .foreachBatch(process_sales)
                    .trigger(availableNow = True)
                    .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/slv_checkpoints/sales_checkpoint")
                    .start())

query.awaitTermination()